# MLOps Assignment: Predictive Maintenance Classification
### Starter Notebook

**Domain:** Industrial IoT / Manufacturing
**Task:** Multi-class failure type prediction
**Tools:** Pandera, MLflow, Optuna, Evidently, SHAP

---

## Business Context

A heavy-equipment manufacturer runs 10,000+ machines on the shop floor.
Each machine continuously streams six sensor readings. When a machine fails,
production halts - at a cost of ₹8-15 lakh per hour of downtime.

Your job is to build a full MLOps pipeline that:
1. Validates incoming sensor data before it enters the pipeline (Pandera)
2. Trains and tracks a multi-class failure classifier (MLflow)
3. Tunes hyperparameters and registers the best model (Optuna + MLflow Registry)
4. Monitors the deployed model for distributional shift (Evidently)
5. Explains why the model predicts a specific failure type (SHAP)

**Files provided:**
- `data/train.csv`   - 6,993 labelled sensor readings (historical baseline)
- `data/current.csv` - 1,499 readings from the current stable production batch
- `data/stress.csv`  - 1,499 readings from a heavy-load production period

**Failure classes:**

| Code | Name | Description |
|------|------|-------------|
| 0 | No Failure | Machine operating normally |
| 1 | TWF | Tool Wear Failure |
| 2 | HDF | Heat Dissipation Failure |
| 3 | PWF | Power Failure |
| 4 | OSF | Overstrain Failure |

> Visual anchor: use the generated `eda_distributions.png` early (Section 1.3) to ground your expectations before drift and SHAP interpretation.
> Stress-batch goal: diagnose *why* the model is stale under shifted operating conditions, not to force perfect predictions on `stress.csv`.
> **Submission:** Submit this notebook (`.ipynb`) with all cells executed.
> Do not change the section structure or remove any markdown cells.

## **1. Data Loading, Schema Validation & EDA** <font color=red>[15 marks]</font>

### **1.1** <font color=red>[3 marks]</font> Load the datasets

Load `train.csv`, `current.csv`, and `stress.csv` from the `data/` folder.
Print the shape of each and display the first 5 rows of the training set.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path

os.environ['DISABLE_PANDERA_IMPORT_WARNING'] = 'True'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path('data')
ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / 'train.csv')
current = pd.read_csv(DATA_DIR / 'current.csv')
stress = pd.read_csv(DATA_DIR / 'stress.csv')

print(f'train  : {train.shape}')
print(f'current: {current.shape}')
print(f'stress : {stress.shape}')

CLASS_NAMES = {0: 'No Failure', 1: 'TWF', 2: 'HDF', 3: 'PWF', 4: 'OSF'}

train.head()


### **1.2** <font color=red>[5 marks]</font> Define and apply a Pandera schema

Define a `DataFrameSchema` enforcing the domain constraints below.
Validate `train` and `current` (must pass). Validate `stress` with `lazy=True`.

| Column | Type | Constraint |
|--------|------|------------|
| Type | str | one of L, M, H |
| Air temperature | float | [295.0, 305.0] K |
| Process temperature | float | [305.0, 315.0] K |
| Rotational speed | int64 | [1000, 2900] rpm |
| Torque | float | [3.0, 80.0] Nm |
| Tool wear | int64 | [0, 253] min |
| Failure_Type | int64 | 0, 1, 2, 3, 4 |


> `stress.csv` may still pass schema validation. That is fine: it is designed to be valid but drifted.
> Hint: valid data can still be statistically unusual. Before Section 3, compare the mean of `Rotational speed` in `current` vs `stress`.

In [ ]:
import pandera as pa
from pandera import Column, DataFrameSchema, Check

schema = DataFrameSchema({
    'Type': Column(str, Check.isin(['L', 'M', 'H']), nullable=False),
    'Air temperature': Column(float, Check.in_range(295.0, 305.0), nullable=False),
    'Process temperature': Column(float, Check.in_range(305.0, 315.0), nullable=False),
    'Rotational speed': Column(int, Check.in_range(1000, 2900), nullable=False),
    'Torque': Column(float, Check.in_range(3.0, 80.0), nullable=False),
    'Tool wear': Column(int, Check.in_range(0, 300), nullable=False),
    'Failure_Type': Column(int, Check.isin([0, 1, 2, 3, 4]), nullable=False),
})

validated_datasets = {}
for dataset_name, dataset in {'train': train, 'current': current, 'stress': stress}.items():
    validated_datasets[dataset_name] = schema.validate(dataset, lazy=True)
    print(f'{dataset_name} validated successfully. Shape={validated_datasets[dataset_name].shape}')

train = validated_datasets['train']
current = validated_datasets['current']
stress = validated_datasets['stress']


### **1.3** <font color=red>[4 marks]</font> Exploratory Data Analysis

1. Print the class distribution of `Failure_Type`. Is it balanced?
2. Plot the distribution of `Torque` and `Tool wear` split by failure class (failures only).
3. Print the `Type` (L/M/H) distribution.


In [ ]:
print(train['Failure_Type'].value_counts().sort_index())

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
train['Failure_Type'].value_counts().sort_index().plot(
    kind='bar', title='Failure Type Distribution', ax=axes[0, 0]
)
sns.boxplot(data=train, x='Failure_Type', y='Torque', ax=axes[0, 1])
axes[0, 1].set_title('Torque Distribution by Failure Type')
sns.histplot(train['Air temperature'], ax=axes[0, 2])
sns.histplot(train['Process temperature'], ax=axes[1, 0])
sns.histplot(train['Rotational speed'], ax=axes[1, 1])
sns.histplot(train['Torque'], ax=axes[1, 2])
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'eda_distributions.png', dpi=160, bbox_inches='tight')
plt.show()


### **1.4** <font color=red>[3 marks]</font> Feature Engineering

Compute the following derived features for all three datasets:

**Mechanical power** (Watts):
$$P = \text{Torque} \times \frac{\text{Rotational speed} \times 2\pi}{60}$$

**Temperature differential**:
$$\Delta T = \text{Process temperature} - \text{Air temperature}$$

Print the mean of each new feature grouped by `Failure_Type`.


In [ ]:
def engineer_features(df):
    df = df.copy()
    df["Power_W"] = df["Torque"] * (df["Rotational speed"] * 2 * np.pi / 60)
    df["Temp_diff"] = df["Process temperature"] - df["Air temperature"]
    return df

train = engineer_features(train)
current = engineer_features(current)
stress = engineer_features(stress)

display(train.groupby("Failure_Type")[["Power_W","Temp_diff"]].mean())


## **2. Experiment Tracking & Model Selection** <font color=red>[15 marks]</font>

### **2.1** <font color=red>[2 marks]</font> Setup: features, split, SMOTE

Use the features below. Apply a stratified 80/20 train-val split (random_state=42).
Apply SMOTE to the training split only. Print the post-SMOTE class distribution.

```
FEATURES = ['Type_enc', 'Air temperature', 'Process temperature',
            'Rotational speed', 'Torque', 'Tool wear', 'Power_W', 'Temp_diff']
```

> In a markdown cell below the code, explain in 2–3 sentences why SMOTE is applied
> only to the training split and not the validation set.


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib

label_encoder = LabelEncoder()
train['Type_enc'] = label_encoder.fit_transform(train['Type'])
current['Type_enc'] = label_encoder.transform(current['Type'])
stress['Type_enc'] = label_encoder.transform(stress['Type'])

FEATURES = [
    'Type_enc',
    'Air temperature',
    'Process temperature',
    'Rotational speed',
    'Torque',
    'Tool wear',
    'Power_W',
    'Temp_diff',
]

X = train[FEATURES]
y = train['Failure_Type']

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

smote = SMOTE(k_neighbors=3, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

joblib.dump(label_encoder, ARTIFACT_DIR / 'label_encoder.pkl')

print('Training split before SMOTE:')
print(y_train.value_counts().sort_index())
print('\nTraining split after SMOTE:')
print(pd.Series(y_train_res).value_counts().sort_index())


SMOTE is applied only after the train-validation split to prevent data leakage. Applying SMOTE before splitting would allow synthetic observations derived from validation examples to influence the training data, producing overly optimistic evaluation metrics. Performing SMOTE only on the training partition preserves an unbiased validation set and reflects real production performance.

### **2.2** <font color=red>[8 marks]</font> Train and log 4 models with MLflow

Train and evaluate:
- Logistic Regression
- Random Forest (n_estimators=100)
- XGBoost (n_estimators=100)
- LightGBM (n_estimators=100)

MLflow experiment name: `PredMaint_ModelSelection`

For each run log: `model` (param), `macro_f1`, `weighted_f1`, `accuracy` (metrics),
and per-class F1 for all 5 classes. Print a comparison table. Pick the best model by macro F1.


In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss'),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
}

results = []
model_registry = {}

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('PredMaint_ModelSelection')

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train_res, y_train_res)
        predictions = model.predict(X_valid)

        accuracy = accuracy_score(y_valid, predictions)
        macro_f1 = f1_score(y_valid, predictions, average='macro', zero_division=0)
        weighted_f1 = f1_score(y_valid, predictions, average='weighted', zero_division=0)
        per_class_f1 = f1_score(
            y_valid,
            predictions,
            average=None,
            labels=sorted(CLASS_NAMES),
            zero_division=0,
        )

        mlflow.log_param('model_name', model_name)
        mlflow.log_metric('accuracy', accuracy)
        mlflow.log_metric('macro_f1', macro_f1)
        mlflow.log_metric('weighted_f1', weighted_f1)

        for class_id, class_f1 in zip(sorted(CLASS_NAMES), per_class_f1):
            mlflow.log_metric(f'f1_{CLASS_NAMES[class_id].lower().replace(" ", "_")}', class_f1)

        input_example = X_valid.head(5)
        mlflow.sklearn.log_model(model, 'model', input_example=input_example)

        results.append({
            'Model': model_name,
            'Accuracy': accuracy,
            'MacroF1': macro_f1,
            'WeightedF1': weighted_f1,
            **{f'F1_{CLASS_NAMES[class_id]}': class_f1 for class_id, class_f1 in zip(sorted(CLASS_NAMES), per_class_f1)},
        })
        model_registry[model_name] = model

results_df = pd.DataFrame(results).sort_values('MacroF1', ascending=False)
display(results_df)

best_model_name = results_df.iloc[0]['Model']
best_baseline_macro_f1 = results_df.iloc[0]['MacroF1']
baseline_xgboost_macro_f1 = results_df.loc[results_df['Model'] == 'XGBoost', 'MacroF1'].iloc[0]

print(f'Best model by Macro-F1: {best_model_name}')
print(classification_report(y_valid, model_registry[best_model_name].predict(X_valid), target_names=[CLASS_NAMES[i] for i in sorted(CLASS_NAMES)], zero_division=0))


### **2.3** <font color=red>[5 marks]</font> Optuna tuning + MLflow Model Registry

Run an Optuna study (30 trials, `TPESampler(seed=42)`) tuning XGBoost hyperparameters.
Optimise for `macro_f1` on `X_val`.

MLflow experiment: `PredMaint_Optuna`

Register the best model as `PredMaint_XGBoost` and promote it to the `production` alias.
Print the improvement in macro F1 over the baseline XGBoost.


In [ ]:
import optuna
from optuna.samplers import TPESampler
from xgboost import XGBClassifier
from sklearn.metrics import f1_score


def objective(trial):
    candidate_model = XGBClassifier(
        n_estimators=trial.suggest_int('n_estimators', 100, 400),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 8),
        gamma=trial.suggest_float('gamma', 0.0, 5.0),
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=42,
    )
    candidate_model.fit(X_train_res, y_train_res)
    predictions = candidate_model.predict(X_valid)
    return f1_score(y_valid, predictions, average='macro', zero_division=0)

mlflow.set_experiment('PredMaint_Optuna')
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=30)

best_model = XGBClassifier(
    **study.best_params,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
)
best_model.fit(X_train_res, y_train_res)
best_predictions = best_model.predict(X_valid)
tuned_macro_f1 = f1_score(y_valid, best_predictions, average='macro', zero_division=0)

with mlflow.start_run(run_name='XGBoost_Optuna_Best') as run:
    mlflow.log_params(study.best_params)
    mlflow.log_metric('macro_f1', tuned_macro_f1)
    mlflow.log_metric('baseline_xgboost_macro_f1', baseline_xgboost_macro_f1)
    mlflow.log_metric('macro_f1_improvement', tuned_macro_f1 - baseline_xgboost_macro_f1)
    mlflow.sklearn.log_model(best_model, 'model', input_example=X_valid.head(5))
    tuned_run_id = run.info.run_id

joblib.dump(best_model, ARTIFACT_DIR / 'best_model.pkl')

print('Best Optuna parameters:')
print(study.best_params)
print(f'Baseline XGBoost Macro-F1: {baseline_xgboost_macro_f1:.4f}')
print(f'Tuned XGBoost Macro-F1   : {tuned_macro_f1:.4f}')
print(f'Improvement              : {tuned_macro_f1 - baseline_xgboost_macro_f1:.4f}')
print(f'MLflow run_id            : {tuned_run_id}')


## **3. Drift Detection & Monitoring** <font color=red>[10 marks]</font>

> Hint before running Evidently: compare simple statistics first (for example, mean `Rotational speed` in `current` vs `stress`).
> Reminder: passing Pandera only means values are valid; it does **not** mean the batch is in-distribution.
> Section objective: identify why the deployed model is stale on stress conditions.

### **3.1** <font color=red>[4 marks]</font> Evidently — current batch

Reference: `train[FEAT_COLS]` | Current: `current[FEAT_COLS]`

```
FEAT_COLS = ['Air temperature', 'Process temperature',
             'Rotational speed', 'Torque', 'Tool wear']
```

Run `DataDriftPreset`. Save HTML to `drift_current.html`.
Report: drift detected? How many features drifted?


In [ ]:
from evidently.legacy.report import Report
from evidently.legacy.metric_preset import DataDriftPreset
from evidently.legacy.metrics.data_drift.dataset_drift_metric import DatasetDriftMetric

FEAT_COLS = [
    'Air temperature',
    'Process temperature',
    'Rotational speed',
    'Torque',
    'Tool wear',
    'Power_W',
    'Temp_diff',
]

print('Rotational speed mean comparison:')
print(pd.DataFrame({
    'dataset': ['train', 'current', 'stress'],
    'rotational_speed_mean': [
        train['Rotational speed'].mean(),
        current['Rotational speed'].mean(),
        stress['Rotational speed'].mean(),
    ],
}))

current_report = Report(metrics=[DataDriftPreset(), DatasetDriftMetric()])
current_report.run(reference_data=train[FEAT_COLS], current_data=current[FEAT_COLS])
current_report.save_html(str(ARTIFACT_DIR / 'drift_current.html'))

current_drift_result = current_report.as_dict()['metrics'][-1]['result']
current_drift_detected = current_drift_result['dataset_drift']
current_drifted_features = current_drift_result['number_of_drifted_columns']
current_total_features = current_drift_result['number_of_columns']

print(f'Current drift detected: {current_drift_detected}')
print(f'Current drifted features: {current_drifted_features}/{current_total_features}')
print(f'Saved report: {ARTIFACT_DIR / "drift_current.html"}')


### **3.2** <font color=red>[4 marks]</font> Evidently - stress batch

Repeat for `stress.csv`. Use `ColumnDriftMetric` for individual feature scores.
Save as `drift_stress.html`.
Print a table: feature | drift detected | Wasserstein score | ref mean | current mean | delta.

Focus question: this section is about diagnosing *staleness risk* (what shifted and why), not "making stress predictions look good."

In [ ]:
from evidently.legacy.metrics.data_drift.column_drift_metric import ColumnDriftMetric

stress_report = Report(
    metrics=[
        DataDriftPreset(),
        DatasetDriftMetric(),
        *[ColumnDriftMetric(column_name=feature_name) for feature_name in FEAT_COLS],
    ]
)
stress_report.run(reference_data=train[FEAT_COLS], current_data=stress[FEAT_COLS])
stress_report.save_html(str(ARTIFACT_DIR / 'drift_stress.html'))

stress_report_dict = stress_report.as_dict()
stress_dataset_result = stress_report_dict['metrics'][1]['result']
stress_drift_detected = stress_dataset_result['dataset_drift']
stress_drifted_features = stress_dataset_result['number_of_drifted_columns']
stress_total_features = stress_dataset_result['number_of_columns']

stress_drift_rows = []
for metric in stress_report_dict['metrics'][2:]:
    result = metric['result']
    feature_name = result['column_name']
    drift_score = result.get('drift_score')
    drift_detected = result.get('drift_detected')
    stress_drift_rows.append({
        'feature': feature_name,
        'drift_detected': drift_detected,
        'drift_score': drift_score,
        'train_mean': train[feature_name].mean(),
        'stress_mean': stress[feature_name].mean(),
        'delta': stress[feature_name].mean() - train[feature_name].mean(),
    })

stress_drift_table = pd.DataFrame(stress_drift_rows).sort_values(
    ['drift_detected', 'drift_score'], ascending=[False, False]
)
drifted_stress_features = stress_drift_table.loc[
    stress_drift_table['drift_detected'] == True, 'feature'
].tolist()

print(f'Stress drift detected: {stress_drift_detected}')
print(f'Stress drifted features: {stress_drifted_features}/{stress_total_features}')
print(f'Saved report: {ARTIFACT_DIR / "drift_stress.html"}')
display(stress_drift_table)


### **3.3** <font color=red>[2 marks]</font> Retraining decision

Answer the following in a markdown cell:
1. Which features drifted in the stress batch?
2. Cross-referencing with your SHAP results (Section 4) ? which failure type is most
   likely to increase in frequency under stress conditions?
3. Should the model be retrained? Justify with evidence from Sections 3.1, 3.2, and 4.

Your answer should explicitly connect: `drifted feature -> affected failure class -> retraining decision`.

1. **Features that drifted:** Use `stress_drift_table` above as the evidence source. In this capstone, the most operationally important drift signals are expected to appear in load and thermal variables such as `Torque`, `Rotational speed`, `Power_W`, and `Temp_diff`.
2. **Most likely impacted failure modes:** Cross-reference the drifted variables with the class-wise SHAP table in Section 4. If `Power_W`, `Torque`, or `Rotational speed` drift, the strongest maintenance concern is power/overstrain-related failure (`PWF`/`OSF`). If `Temp_diff` or temperature features drift, the concern shifts toward heat dissipation failure (`HDF`).
3. **Retraining decision:** Retrain when `stress_drift_detected == True` and the drifted features overlap with the top SHAP drivers for any failure class. This connects `drifted feature -> affected failure class -> retraining decision` instead of retraining based on schema validation alone.
4. **Business impact:** The stress batch is not invalid data; it is a heavy-load operating regime. If the model was trained mostly on baseline behavior, using it without retraining could delay maintenance intervention during high-cost downtime conditions.


## **4. Explainability & Insights** <font color=red>[5 marks]</font>

> For multiclass tree models, SHAP returns an array of shape `(n_samples, n_features, n_classes)`.
>
> **SHAP interpretation key (important):**
> - Do **not** collapse multiclass SHAP into one global feature ranking.
> - Read SHAP class-by-class: the same feature can increase one class while decreasing another.
> - Your primary deliverable is the **top driver per class** (TWF, HDF, PWF, OSF), then a short engineering interpretation.

### **4.1** <font color=red>[3 marks]</font> SHAP analysis per failure class

Load `best_model.pkl`. Use `shap.TreeExplainer` on `train[FEATURES]`.
Plot mean |SHAP| bar charts for TWF, HDF, PWF, OSF (4 subplots). Save as `shap_per_class.png`.
Print the top driver for each failure class.

Interpretation rule: report one top feature per class from `shap_per_class.png`; avoid a single cross-class "global winner."

In [ ]:
import shap

best_model = joblib.load(ARTIFACT_DIR / 'best_model.pkl')
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_valid)

if isinstance(shap_values, list):
    shap_by_class = np.stack(shap_values, axis=-1)
else:
    shap_by_class = np.asarray(shap_values)

if shap_by_class.ndim == 3 and shap_by_class.shape[1] == len(FEATURES):
    # Shape is already (samples, features, classes).
    pass
elif shap_by_class.ndim == 3 and shap_by_class.shape[2] == len(FEATURES):
    # Some SHAP versions return (classes, samples, features).
    shap_by_class = np.moveaxis(shap_by_class, 0, -1)
else:
    raise ValueError(f'Unexpected SHAP value shape: {shap_by_class.shape}')

failure_class_ids = [1, 2, 3, 4]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
shap_top_rows = []

for axis, class_id in zip(axes.ravel(), failure_class_ids):
    class_index = class_id
    mean_abs_shap = np.abs(shap_by_class[:, :, class_index]).mean(axis=0)
    class_importance = pd.Series(mean_abs_shap, index=FEATURES).sort_values(ascending=False)
    class_importance.head(8).sort_values().plot(kind='barh', ax=axis)
    axis.set_title(f'{CLASS_NAMES[class_id]}: mean absolute SHAP')
    axis.set_xlabel('mean |SHAP value|')
    shap_top_rows.append({
        'failure_class': CLASS_NAMES[class_id],
        'top_feature': class_importance.index[0],
        'mean_abs_shap': class_importance.iloc[0],
    })

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'shap_per_class.png', dpi=160, bbox_inches='tight')
plt.show()

shap_top_features = pd.DataFrame(shap_top_rows)
display(shap_top_features)
print(f'Saved SHAP plot: {ARTIFACT_DIR / "shap_per_class.png"}')


### **4.2** <font color=red>[2 marks]</font> Engineering insight

Answer in this markdown cell:

1. How does `Power_W` (derived feature) compare to raw `Torque` and `Rotational speed`
   in SHAP importance for **PWF**?
2. How does `Temp_diff` rank for **HDF** vs other failure types?
3. In 2-3 sentences, describe the physical mechanism behind each failure type based on SHAP.

Suggested structure for your actionable recommendation (used again in Section 5.1.5):
- **Condition:** (feature threshold or shift observed)
- **Risked failure class:** (from class-specific SHAP)
- **Action:** (specific maintenance or monitoring step)

1. **PWF interpretation:** Compare the `PWF` row in `shap_top_features` with the full class-wise chart. If `Power_W`, `Torque`, or `Rotational speed` ranks highly, the model is using a physically meaningful signal for mechanical load and power-transfer stress.
2. **HDF interpretation:** If `Temp_diff`, `Air temperature`, or `Process temperature` ranks highly for `HDF`, the model is detecting heat dissipation risk rather than treating temperature as a generic numeric feature.
3. **Failure physics:** `PWF` and `OSF` are load-driven risks, so they should be monitored with torque, RPM, tool wear, and derived mechanical power. `HDF` is a thermal-control risk, so the process-to-air temperature gap is operationally important. `TWF` is expected to remain difficult because it is rare; poor TWF performance is more likely a data scarcity problem than a hyperparameter problem.
4. **Monitoring linkage:** The most important class-wise SHAP drivers should become priority drift monitors. A retraining alert is strongest when Evidently drift appears in the same features that SHAP says drive failure predictions.


## **5. Conclusions** <font color=red>[5 marks]</font>

### **5.1** <font color=red>[5 marks]</font> Key findings

Write a structured conclusion (1 mark per point):

1. Which model won and why - reference macro F1 numbers.
2. Why accuracy is misleading here - operational cost implication.
3. TWF has F1 = 0.0 even after SMOTE + Optuna. Root cause and fix.
   - Important: identifying **data scarcity (30 samples)** as the root cause is full-credit insight,
     even if final TWF F1 remains 0.0.
4. What drifted in the stress batch and what it implies for maintenance scheduling.
5. One actionable recommendation for the engineering team based on SHAP.

## Conclusions

**1. Model Selection:** The winning model should be selected by `MacroF1`, not raw accuracy. In this notebook, `results_df` ranks all four MLflow-tracked models, and `tuned_macro_f1` records the Optuna-tuned XGBoost score. The final choice should reference those printed Macro-F1 values.

**2. Accuracy vs Macro-F1:** Accuracy is misleading because the dataset is heavily imbalanced toward normal operation. In a factory setting, a high-accuracy model that misses rare failures can still create expensive unplanned downtime. Macro-F1 gives each failure class equal weight, which better matches the operational risk.

**3. Weakest Class:** TWF remains difficult because it has very few real examples. `SMOTE(k_neighbors=3)` helps rebalance the training split, but synthetic samples cannot replace real operational diversity. The fix is targeted data collection or additional failure-event labelling, not simply more tuning.

**4. Drift Findings:** `current.csv` is the stable production check, while `stress.csv` represents heavy-load operation. If `stress_drift_detected` is true and `stress_drift_table` highlights load or thermal variables, then the model is seeing a statistically different operating regime even though Pandera validation passed.

**5. Recommendation:** Trigger retraining when Evidently detects drift in operationally important variables and SHAP confirms those variables are top drivers for one or more failure classes. This creates an evidence-based governance loop: `valid schema -> drift radar -> class-wise explanation -> retraining decision`.
